# Fashion-MNIST 图像分类：MLP 与 CNN

环境


In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from pathlib import Path
import time

print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

**Step 1｜数据准备**
- 定位数据在哪；
- 数据预处理张量，标准化；
- 分一下训练集和测试集；
- 展示图片；

In [ ]:
DATA_ROOT = Path.cwd()
while not(DATA_ROOT / "data").exists() and DATA_ROOT.parent != DATA_ROOT:
    DATA_ROOT = DATA_ROOT.parent

DATA_DIR = DATA_ROOT/"data"
mean = (0.2860,)
std = (0.3530,)
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(mean,std)]
)

train_ds = datasets.FashionMNIST(
    root=DATA_DIR,
    train=True,
    download=False,
    transform=transform
)
test_ds = datasets.FashionMNIST(
    root=DATA_DIR,
    train=False,
    download=False,
    transform=transform
)

classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

fig, ax = plt.subplots(1,5,figsize=(10,3))
for i in range(5):
    image,label = train_ds[i]
    display_image = image * std[0] + mean[0]
    ax[i].imshow(display_image.squeeze(),cmap="gray")
    ax[i].set_title(classes[label])
    ax[i].axis("off")
plt.tight_layout()
plt.show()

- 分批图片：训练集图片顺序打乱，拿来训练；测试集不用

**Step 2｜MLP**
- 建一个MLP模型：Flatten+两层连接层+激活函数

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
)

images, labels = next(iter(train_loader))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential (
        nn.Flatten(),
        nn.Linear(28*28, 256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

mlp = MLP()
mlp_logits = mlp(images)

- 损失函数，优化器
- 训练函数
- 评估函数

In [ ]:
loss_fn = nn.CrossEntropyLoss()

mlp_optimizer = torch.optim.Adam(
    mlp.parameters(),
    lr = 0.001
)

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for x,y in loader:
        optimizer.zero_grad()

        logits = model(x)
        loss = loss_fn(logits,y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()* y.size(0)
        correct += (logits.argmax(dim = 1) == y).sum().item()
        total += y.size(0)

    return total_loss/total , correct/total

def evaluate(model, loader, loss_fn):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in loader:
            logits = model(x)
            loss = loss_fn(logits,y)

            total_loss += loss.item()* y.size(0)
            correct += (logits.argmax(dim = 1) == y).sum().item()
            total += y.size(0)

    return total_loss/total , correct/total

- 训练MLP

In [ ]:
epochs = 5

mlp_train_losses, mlp_train_accs = [], []
mlp_test_losses, mlp_test_accs = [], []

start = time.perf_counter()

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(mlp, train_loader, loss_fn, mlp_optimizer)
    test_loss, test_acc = evaluate(mlp, test_loader, loss_fn)

    mlp_train_losses.append(train_loss)
    mlp_train_accs.append(train_acc)
    mlp_test_losses.append(test_loss)
    mlp_test_accs.append(test_acc)

    print(
        f"epoch {epoch+1}/{epochs} ："
        f"\n\t训练loss: {train_loss:.4f} 训练acc: {train_acc:.4f}"
        f"\n\t测试loss: {test_loss:.4f} 测试acc: {test_acc:.4f}"
    )

mlp_time = time.perf_counter() - start
print(f"用时：{mlp_time:.4f}s")


In [ ]:
total_params = sum(p.numel() for p in mlp.parameters())
print(f"总参数量: {total_params:,} 个")

画图

In [ ]:
epochs_range = range(1,epochs+1)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.plot(epochs_range,mlp_train_losses,marker = "o",label="train loss")
plt.plot(epochs_range, mlp_test_losses, marker ="o", label="test loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.title("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1,2,2)
plt.plot(epochs_range, mlp_train_accs, marker ="o", label="train acc")
plt.plot(epochs_range,mlp_test_accs,marker = "o",label="test acc")
plt.xlabel("epochs")
plt.ylabel("acc")
plt.title("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Step 3｜CNN**

其实有点类似在 MLP 基础上加了个处理头，卷积核参数初始随机，在优化器帮助下逐渐收敛
1. 第一层卷积层和池化层：将 images.shape 变形为 (128,32,14,14)
2. 第二层卷积层和池化层：(128,32,14,14) 变形为 (128,64,7,7) 的特征图
3. 分类得分：    把特征图 Flatten 一下，拿去给 Linear 跑分，输出 10 个标签

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential (
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn = CNN()
cnn_logits = cnn(images)


- 训练

In [ ]:
epochs = 5

cnn_train_losses, cnn_train_accs = [], []
cnn_test_losses, cnn_test_accs = [], []

cnn_optimizer = torch.optim.Adam(
    cnn.parameters(),
    lr = 0.001
)
start = time.perf_counter()

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(cnn, train_loader, loss_fn, cnn_optimizer)
    test_loss, test_acc = evaluate(cnn, test_loader, loss_fn)

    cnn_train_losses.append(train_loss)
    cnn_train_accs.append(train_acc)
    cnn_test_losses.append(test_loss)
    cnn_test_accs.append(test_acc)

    print(
        f"epoch {epoch+1}/{epochs} ："
        f"\n\t训练loss: {train_loss:.4f} 训练acc: {train_acc:.4f}"
        f"\n\t测试loss: {test_loss:.4f} 测试acc: {test_acc:.4f}"
    )

cnn_time = time.perf_counter() - start
print(f"用时：{cnn_time:.4f}s")


In [ ]:
total_params = sum(p.numel() for p in cnn.parameters())
print(f"总参数量: {total_params:,} 个")

- 画图

In [ ]:
epochs_range = range(1,epochs+1)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.plot(epochs_range,cnn_train_losses,marker = "o",label="train loss")
plt.plot(epochs_range, cnn_test_losses, marker ="o", label="test loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.title("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1,2,2)
plt.plot(epochs_range, cnn_train_accs, marker ="o", label="train acc")
plt.plot(epochs_range,cnn_test_accs,marker = "o",label="test acc")
plt.xlabel("epochs")
plt.ylabel("acc")
plt.title("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

- CNN的卷积可视图（以第一张图为例）
- 可以跳过ReLU,因为只是可视化特征图，不需要打分

In [ ]:
image,label = test_ds[0]
x = image.unsqueeze(0)

with torch.no_grad():
    x1 = cnn.features[0](x)
    x2 = cnn.features[2](x1)
    x3 = cnn.features[3](x2)
    x4 = cnn.features[5](x3)

stages = [("conv1", x1), ("pool1", x2), ("conv2", x3), ("pool2", x4)]

fig, axes = plt.subplots(4, 8, figsize=(8, 5))
for r in range(4):
    name, fmap = stages[r]
    for c in range(8):
        axes[r, c].imshow(fmap[0, c].numpy(), cmap="gray")
        axes[r, c].axis("off")
    axes[r, 0].set_title(name)

plt.tight_layout()
plt.show()

**Step 4｜对比分析**
1. **比较两种模型的准确率、参数量、训练时间**

    | 模型 |  测试准确率 | 参数量 |     训练时间 |
    | --- |-------:| ---: |---------:|
    | MLP | 88.51% | 235,146 |  81.75 s |
    | CNN | 91.28% | 421,642 | 286.22 s |

- CNN的参数量和训练时间明显比MLP多，但也换来了比MLP高三个百分点的成绩。

2. **分析CNN相比MLP取得更好成绩的原因**
- **局部连接**：只看附近的像素，不像MLP全连接，有利于识别空间结构规律，所需参数量也会变少。
- **权值共享**：步长为 1 时，同一个卷积核会在同一层的不同位置重复使用，因此能检测平移后的相似模式，就是平移不变性。而且相比MLP也节省了参数量。
- **池化**：   把长宽减半，保留主要响应，减少一下计算量。

- 如果把同样的参数量给MLP，它的表现会不会更好？
- 顺便造一个4层的MLP看看

In [ ]:
class MLP_Pro(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 400),
            nn.ReLU(),
            nn.Linear(400, 200),
            nn.ReLU(),
            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 10)
        )

    def forward(self, x):
        return self.features(x)

mlp_pro = MLP_Pro()
total_params = sum(p.numel() for p in  mlp_pro.parameters())
print(f"总参数量: {total_params:,} 个")

- 419860，和CNN就差2000左右，很不错的估算结果了。跑5轮看看表现如何

In [ ]:
epochs = 5

mlp_pro_train_losses, mlp_pro_train_accs = [], []
mlp_pro_test_losses, mlp_pro_test_accs = [], []

mlp_pro_optimizer = torch.optim.Adam(
    mlp_pro.parameters(),
    lr = 0.001
)

start = time.perf_counter()

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(mlp_pro, train_loader, loss_fn, mlp_pro_optimizer)
    test_loss, test_acc = evaluate(mlp_pro, test_loader, loss_fn)

    mlp_pro_train_losses.append(train_loss)
    mlp_pro_train_accs.append(train_acc)
    mlp_pro_test_losses.append(test_loss)
    mlp_pro_test_accs.append(test_acc)

    print(
        f"epoch {epoch+1}/{epochs} ："
        f"\n\t训练loss: {train_loss:.4f} 训练acc: {train_acc:.4f}"
        f"\n\t测试loss: {test_loss:.4f} 测试acc: {test_acc:.4f}"
    )

mlp_pro_time = time.perf_counter() - start
print(f"用时：{mlp_pro_time:.4f}s")

- 总之还是比不过，而且和原本的也差不太多，说明在这个数据集上MLP不做特化，估计就是比不过CNN的。
- 然而MLP是干数据处理的，在CNN的强项上比赛肯定不占好处，在现实工业界运用中，还是得对症下药。

**总结**

- MLP，主要思路就是把数据集拆碎，然后用linear死算穷举。这样做在作业1中找房产价格规律取得了不错的成绩，不过全连接的思维对于图像的空间逻辑就基本处理不了，对着拆碎的一堆像素去想空间规律纯粹吃力不讨好。
- CNN，多了个卷积核之后，引入了局部连接和权值共享的架构，非常适合识别图像的空间结构，在本次实验中也取得了不错的表现。